# Day 4 – Feature Engineering for Forecasting

## Objective

The objective of Day 4 is to create model-ready forecasting features from the daily sales dataset.

This includes lag features, rolling averages, time-based features, and a final dataset that can be used for machine learning forecasting models.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

pd.set_option("display.max_columns", None)

print("Libraries imported successfully.")

Libraries imported successfully.


In [2]:
daily_sales = pd.read_csv("../data/processed/daily_sales.csv")

daily_sales["Order Date"] = pd.to_datetime(daily_sales["Order Date"])

print("Daily sales dataset loaded successfully.")
print(daily_sales.shape)

daily_sales.head()

Daily sales dataset loaded successfully.
(1230, 2)


,Order Date,Sales
0,2015-01-03,16.448
1,2015-01-04,288.060
2,2015-01-05,19.536
3,2015-01-06,4407.100
4,2015-01-07,87.158


In [3]:
feature_df = daily_sales.copy()

feature_df["Year"] = feature_df["Order Date"].dt.year
feature_df["Month"] = feature_df["Order Date"].dt.month
feature_df["Quarter"] = feature_df["Order Date"].dt.quarter
feature_df["Week"] = feature_df["Order Date"].dt.isocalendar().week.astype(int)
feature_df["Day"] = feature_df["Order Date"].dt.day
feature_df["DayOfWeek"] = feature_df["Order Date"].dt.dayofweek
feature_df["DayOfYear"] = feature_df["Order Date"].dt.dayofyear
feature_df["IsWeekend"] = (
    feature_df["DayOfWeek"] >= 5
).astype(int)

feature_df.head()

,Order Date,Sales,Year,Month,Quarter,Week,Day,DayOfWeek,DayOfYear,IsWeekend
0,2015-01-03,16.448,2015,1,1,1,3,5,3,1
1,2015-01-04,288.060,2015,1,1,1,4,6,4,1
2,2015-01-05,19.536,2015,1,1,2,5,0,5,0
3,2015-01-06,4407.100,2015,1,1,2,6,1,6,0
4,2015-01-07,87.158,2015,1,1,2,7,2,7,0


In [4]:
print(feature_df.columns)

feature_df.head()

Index(['Order Date', 'Sales', 'Year', 'Month', 'Quarter', 'Week', 'Day',
       'DayOfWeek', 'DayOfYear', 'IsWeekend'],
      dtype='str')


,Order Date,Sales,Year,Month,Quarter,Week,Day,DayOfWeek,DayOfYear,IsWeekend
0,2015-01-03,16.448,2015,1,1,1,3,5,3,1
1,2015-01-04,288.060,2015,1,1,1,4,6,4,1
2,2015-01-05,19.536,2015,1,1,2,5,0,5,0
3,2015-01-06,4407.100,2015,1,1,2,6,1,6,0
4,2015-01-07,87.158,2015,1,1,2,7,2,7,0


In [5]:
feature_df["Lag_1"] = feature_df["Sales"].shift(1)
feature_df["Lag_7"] = feature_df["Sales"].shift(7)
feature_df["Lag_30"] = feature_df["Sales"].shift(30)

feature_df.head(35)

,Order Date,Sales,Year,Month,Quarter,Week,Day,DayOfWeek,DayOfYear,IsWeekend,Lag_1,Lag_7,Lag_30
0,2015-01-03,16.448,2015,1,1,1,3,5,3,1,NaN,NaN,NaN
1,2015-01-04,288.060,2015,1,1,1,4,6,4,1,16.448,NaN,NaN
2,2015-01-05,19.536,2015,1,1,2,5,0,5,0,288.060,NaN,NaN
3,2015-01-06,4407.100,2015,1,1,2,6,1,6,0,19.536,NaN,NaN
4,2015-01-07,87.158,2015,1,1,2,7,2,7,0,4407.100,NaN,NaN
5,2015-01-09,40.544,2015,1,1,2,9,4,9,0,87.158,NaN,NaN
6,2015-01-10,54.830,2015,1,1,2,10,5,10,1,40.544,NaN,NaN
7,2015-01-11,9.940,2015,1,1,2,11,6,11,1,54.830,16.448,NaN
8,2015-01-13,3553.795,2015,1,1,3,13,1,13,0,9.940,288.060,NaN
9,2015-01-14,61.960,2015,1,1,3,14,2,14,0,3553.795,19.536,NaN


In [6]:
print(feature_df[["Lag_1", "Lag_7", "Lag_30"]].isnull().sum())

feature_df.head(35)

Lag_1      1
Lag_7      7
Lag_30    30
dtype: int64


,Order Date,Sales,Year,Month,Quarter,Week,Day,DayOfWeek,DayOfYear,IsWeekend,Lag_1,Lag_7,Lag_30
0,2015-01-03,16.448,2015,1,1,1,3,5,3,1,NaN,NaN,NaN
1,2015-01-04,288.060,2015,1,1,1,4,6,4,1,16.448,NaN,NaN
2,2015-01-05,19.536,2015,1,1,2,5,0,5,0,288.060,NaN,NaN
3,2015-01-06,4407.100,2015,1,1,2,6,1,6,0,19.536,NaN,NaN
4,2015-01-07,87.158,2015,1,1,2,7,2,7,0,4407.100,NaN,NaN
5,2015-01-09,40.544,2015,1,1,2,9,4,9,0,87.158,NaN,NaN
6,2015-01-10,54.830,2015,1,1,2,10,5,10,1,40.544,NaN,NaN
7,2015-01-11,9.940,2015,1,1,2,11,6,11,1,54.830,16.448,NaN
8,2015-01-13,3553.795,2015,1,1,3,13,1,13,0,9.940,288.060,NaN
9,2015-01-14,61.960,2015,1,1,3,14,2,14,0,3553.795,19.536,NaN


In [7]:
feature_df["Rolling_Mean_7"] = (
    feature_df["Sales"]
    .rolling(window=7)
    .mean()
)

feature_df["Rolling_Mean_30"] = (
    feature_df["Sales"]
    .rolling(window=30)
    .mean()
)

feature_df.head(35)

,Order Date,Sales,Year,Month,Quarter,Week,Day,DayOfWeek,DayOfYear,IsWeekend,Lag_1,Lag_7,Lag_30,Rolling_Mean_7,Rolling_Mean_30
0,2015-01-03,16.448,2015,1,1,1,3,5,3,1,NaN,NaN,NaN,NaN,NaN
1,2015-01-04,288.060,2015,1,1,1,4,6,4,1,16.448,NaN,NaN,NaN,NaN
2,2015-01-05,19.536,2015,1,1,2,5,0,5,0,288.060,NaN,NaN,NaN,NaN
3,2015-01-06,4407.100,2015,1,1,2,6,1,6,0,19.536,NaN,NaN,NaN,NaN
4,2015-01-07,87.158,2015,1,1,2,7,2,7,0,4407.100,NaN,NaN,NaN,NaN
5,2015-01-09,40.544,2015,1,1,2,9,4,9,0,87.158,NaN,NaN,NaN,NaN
6,2015-01-10,54.830,2015,1,1,2,10,5,10,1,40.544,NaN,NaN,701.953714,NaN
7,2015-01-11,9.940,2015,1,1,2,11,6,11,1,54.830,16.448,NaN,701.024000,NaN
8,2015-01-13,3553.795,2015,1,1,3,13,1,13,0,9.940,288.060,NaN,1167.557571,NaN
9,2015-01-14,61.960,2015,1,1,3,14,2,14,0,3553.795,19.536,NaN,1173.618143,NaN


In [8]:
print(
    feature_df[
        ["Rolling_Mean_7", "Rolling_Mean_30"]
    ].isnull().sum()
)

feature_df.head(35)

Rolling_Mean_7      6
Rolling_Mean_30    29
dtype: int64


,Order Date,Sales,Year,Month,Quarter,Week,Day,DayOfWeek,DayOfYear,IsWeekend,Lag_1,Lag_7,Lag_30,Rolling_Mean_7,Rolling_Mean_30
0,2015-01-03,16.448,2015,1,1,1,3,5,3,1,NaN,NaN,NaN,NaN,NaN
1,2015-01-04,288.060,2015,1,1,1,4,6,4,1,16.448,NaN,NaN,NaN,NaN
2,2015-01-05,19.536,2015,1,1,2,5,0,5,0,288.060,NaN,NaN,NaN,NaN
3,2015-01-06,4407.100,2015,1,1,2,6,1,6,0,19.536,NaN,NaN,NaN,NaN
4,2015-01-07,87.158,2015,1,1,2,7,2,7,0,4407.100,NaN,NaN,NaN,NaN
5,2015-01-09,40.544,2015,1,1,2,9,4,9,0,87.158,NaN,NaN,NaN,NaN
6,2015-01-10,54.830,2015,1,1,2,10,5,10,1,40.544,NaN,NaN,701.953714,NaN
7,2015-01-11,9.940,2015,1,1,2,11,6,11,1,54.830,16.448,NaN,701.024000,NaN
8,2015-01-13,3553.795,2015,1,1,3,13,1,13,0,9.940,288.060,NaN,1167.557571,NaN
9,2015-01-14,61.960,2015,1,1,3,14,2,14,0,3553.795,19.536,NaN,1173.618143,NaN


In [9]:
forecast_df = feature_df.dropna().reset_index(drop=True)

print(forecast_df.shape)

forecast_df.head()

(1200, 15)


,Order Date,Sales,Year,Month,Quarter,Week,Day,DayOfWeek,DayOfYear,IsWeekend,Lag_1,Lag_7,Lag_30,Rolling_Mean_7,Rolling_Mean_30
0,2015-02-14,576.726,2015,2,1,7,14,5,45,1,129.568,97.112,16.448,487.067143,612.546233
1,2015-02-15,21.360,2015,2,1,7,15,6,46,1,576.726,134.384,288.060,470.920857,603.656233
2,2015-02-16,9.040,2015,2,1,8,16,0,47,0,21.360,330.512,19.536,424.996286,603.306367
3,2015-02-17,54.208,2015,2,1,8,17,1,48,0,9.040,180.320,4407.100,406.980286,458.209967
4,2015-02-18,37.784,2015,2,1,8,18,2,49,0,54.208,14.560,87.158,410.298000,456.564167


In [10]:
forecast_df.to_csv(
    "../data/processed/forecast_features.csv",
    index=False
)

print("Forecast feature dataset saved successfully.")

Forecast feature dataset saved successfully.


In [11]:
import os

os.listdir("../data/processed")

['daily_sales.csv',
 'forecast_features.csv',
 'monthly_sales.csv',
 'superstore_sales_processed.csv',
 'vgsales_processed.csv',
 'weekly_sales.csv']

# Day 4 Summary

## Tasks Completed

- Loaded daily sales dataset.
- Created time-based features.
- Created lag features (1, 7, 30 days).
- Created rolling average features (7-day and 30-day).
- Removed missing values.
- Saved final forecasting feature dataset.

## Files Created

data/processed/forecast_features.csv

## Skills Learned

- Time Series Feature Engineering
- Lag Features
- Rolling Window Features
- Data Preparation for Forecasting Models

Day 4 successfully completed.